# Лагерные хроники: интерактивная новелла
Проект на чистом Python в формате одного .ipynb: ООП, ветвления, циклы, списки, множества, словари и запись финала в файл.

In [ ]:
class Character:
    def __init__(self, name, role, mood):
        self.name = name
        self.role = role
        self.mood = mood


class PlayerState:
    def __init__(self, name):
        self.name = name
        self.energy = 3
        self.reputation = 0
        self.courage = 0
        self.inventory = []
        self.clues = set()
        self.route = []
        self.flags = {}
        self.ending = ''


class CampChroniclesGame:
    def __init__(self):
        self.npcs = [
            Character('Алиса', 'лидер отряда', 'подозрительная'),
            Character('Электроник', 'техник', 'взволнованный'),
            Character('Ольга Дмитриевна', 'вожатая', 'строгая')
        ]

        # Готовые словари (по критерию):
        self.camp_places = {
            'площадь': 'Пустая площадь, но у памятника кто-то оставил свежие следы.',
            'сцена': 'За кулисами спрятан список выступлений и чужая записка.',
            'столовая': 'На раздаче говорят, что ночью видели человека в плаще.',
            'лодочная': 'Под досками шатается тайник с ржавым замком.',
            'радиорубка': 'В журнале дежурств вырвана одна страница.'
        }
        self.suspicion_levels = {
            'Алиса': 1,
            'Электроник': 2,
            'Ольга Дмитриевна': 0
        }
        self.state = None

    def choose(self, title, options):
        print('')
        print(title)
        for key in options:
            print(str(key) + '. ' + options[key])
        ans = input('> ').strip()
        while ans not in options:
            print('Введи номер из списка.')
            ans = input('> ').strip()
        return ans

    def intro(self):
        print('Лагерь "Солнечная Заря". Прошла неделя смены, и ночью пропадает архивная коробка с документами.')
        name = input('Имя героя: ').strip()
        if name == '':
            name = 'Семён'
        self.state = PlayerState(name)
        self.state.route.append('Прибытие в лагерь')
        print('Ты — ' + self.state.name + '. Вожатая просит помочь до отбоя найти пропажу.')

    def day_one(self):
        ans = self.choose(
            'Первый шаг:',
            {'1': 'Помочь вожатой собрать показания', '2': 'Сразу пойти по следу в одиночку'}
        )
        if ans == '1':
            self.state.reputation += 2
            self.state.inventory.append('блокнот вожатой')
            self.state.route.append('Помощь вожатой')
            self.state.flags['trusted_by_leader'] = True
        else:
            self.state.courage += 2
            self.state.energy -= 1
            self.state.inventory.append('старый компас')
            self.state.route.append('Одиночный поиск')
            self.state.flags['trusted_by_leader'] = False

    def inspect_camp(self):
        print('')
        print('До отбоя можно проверить только 3 места.')
        checked = 0
        for place in self.camp_places:
            if checked == 3:
                break
            ans = self.choose(
                "Проверить локацию '{}' ?".format(place),
                {'1': 'Да', '2': 'Нет'}
            )
            if ans == '1':
                checked += 1
                self.state.route.append('Осмотр: ' + place)
                self.state.clues.add(place)
                self.state.inventory.append('улика: ' + place)
                print(self.camp_places[place])
                if place == 'лодочная':
                    self.state.courage += 1
                if place == 'столовая':
                    self.state.reputation += 1

        if checked == 0:
            self.state.energy += 1
            self.state.route.append('Герой ничего не проверил')

    def radio_puzzle(self):
        print('')
        print('Электроник говорит: в радиорубке скрыт код доступа к архиву.')
        attempts = 3
        while attempts > 0:
            code = input('Введи 4-значный код: ').strip()
            if code == '1989':
                print('Код верный. В журнале найдено имя дежурного.')
                self.state.clues.add('код-1989')
                self.state.reputation += 1
                self.state.route.append('Радиорубка взломана')
                return True
            attempts -= 1
            print('Код неверный. Осталось попыток: ' + str(attempts))
        self.state.route.append('Радиорубка не взломана')
        return False

    def night_event(self):
        print('')
        print('Ночью ты замечаешь силуэт у склада реквизита.')
        turns = 2
        while turns > 0:
            ans = self.choose(
                'Твое действие:',
                {'1': 'Преследовать силуэт', '2': 'Позвать помощь'}
            )
            if ans == '1':
                self.state.courage += 1
                self.state.energy -= 1
                self.state.route.append('Ночное преследование')
                self.state.clues.add('следы у склада')
                break
            else:
                self.state.reputation += 1
                self.state.route.append('Позвал помощь')
                turns -= 1

        if turns == 0:
            self.state.route.append('Силуэт скрылся')

    def optimize_inventory(self):
        if len(self.state.inventory) > 6:
            self.state.inventory.pop(0)

        if 'старый компас' in self.state.inventory and 'блокнот вожатой' in self.state.inventory:
            self.state.inventory.remove('старый компас')
            self.state.inventory.append('карта маршрутов')

        if 'улика: сцена' in self.state.inventory:
            self.state.clues.add('переписанный сценарий')

    def final_decision(self, radio_ok):
        accuse = self.choose(
            'Кого обвинить на общем сборе?',
            {'1': 'Алису', '2': 'Электроника', '3': 'Никого не обвинять и раскрыть схему'}
        )

        if accuse == '1':
            self.state.route.append('Обвинение Алисы')
            self.suspicion_levels['Алиса'] += 2
        elif accuse == '2':
            self.state.route.append('Обвинение Электроника')
            self.suspicion_levels['Электроник'] += 2
        else:
            self.state.route.append('Обвинений нет, анализ фактов')
            self.state.reputation += 1

        has_many_clues = len(self.state.clues) >= 3
        trusted = self.state.reputation >= 3
        brave = self.state.courage >= 2

        if radio_ok and has_many_clues and trusted and brave and accuse == '3':
            self.state.ending = 'Истинная концовка: ты раскрываешь подмену архивов и спасаешь смену от закрытия.'
        elif has_many_clues and (accuse == '1' or accuse == '2'):
            self.state.ending = 'Драматичная концовка: виновный найден, но часть лагеря считает решение несправедливым.'
        elif radio_ok or trusted:
            self.state.ending = 'Нейтральная концовка: расследование завершено частично, лагерь продолжает жить обычной жизнью.'
        else:
            self.state.ending = 'Плохая концовка: доказательств не хватает, дело закрывают без ответа.'

    def build_summary(self):
        # Новый словарь (по критерию), создается во время выполнения:
        summary = {}
        summary['герой'] = self.state.name
        summary['энергия'] = self.state.energy
        summary['репутация'] = self.state.reputation
        summary['смелость'] = self.state.courage
        summary['маршрут'] = self.state.route
        summary['улики'] = list(self.state.clues)
        summary['инвентарь'] = self.state.inventory
        summary['подозрения'] = self.suspicion_levels
        summary['финал'] = self.state.ending
        return summary

    def save_summary(self, summary):
        lines = []
        lines.append('Герой: ' + summary['герой'])
        lines.append('Энергия: ' + str(summary['энергия']))
        lines.append('Репутация: ' + str(summary['репутация']))
        lines.append('Смелость: ' + str(summary['смелость']))
        lines.append('Маршрут игрока:')
        for step in summary['маршрут']:
            lines.append('- ' + step)
        lines.append('Улики: ' + ', '.join(summary['улики']))
        lines.append('Инвентарь: ' + ', '.join(summary['инвентарь']))
        lines.append('Подозрения: ' + str(summary['подозрения']))
        lines.append('Концовка: ' + summary['финал'])

        with open('novel_result.txt', 'w', encoding='utf-8') as f:
            f.write('\\n'.join(lines))

    def run(self):
        self.intro()
        self.day_one()
        self.inspect_camp()
        radio_ok = self.radio_puzzle()
        self.night_event()
        self.optimize_inventory()
        self.final_decision(radio_ok)
        summary = self.build_summary()
        self.save_summary(summary)
        print('')
        print(self.state.ending)
        print('Итог игры сохранен в novel_result.txt')


game = CampChroniclesGame()
game.run()
